**Базовые методики предобработки**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('Titanic-Dataset.csv')

**1. Выбросы.** Z-оценка и IQR - это статистическая идентификация. Методы обработки: удаление, замена (медиана / среднее), трансформация (логарифм, корень, box-cox, yeo-johnson), кодирование в отдельную категорию. На примере FarePerPerson.

In [3]:
from scipy import stats

# FarePerPerson — из test1: Fare это цена за билет на группу, делим на размер группы
df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")
df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"]
fpp = df["FarePerPerson"]

# Идентификация статистикой
# Z-оценка: на сколько стандартных отклонений значение ушло от среднего
z = (fpp - fpp.mean()) / fpp.std()
print("Выбросов по правилу |Z| > 3:", (z.abs() > 3).sum())

# IQR: выброс — то, что дальше 1.5*IQR от границ «ящика» (Q1 и Q3)
q1, q3 = fpp.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = (fpp < low) | (fpp > high)
print(f"Выбросов по IQR (за пределами [{low:.1f}, {high:.1f}]):", outliers.sum())
print("IQR строже: Z-оценка сама страдает от выбросов (они раздувают std).")

# Методы обработки
# 1. Удаление строк с выбросами
fpp_dropped = fpp[~outliers]

# 2. Замена: выбросы -> медиана (или среднее)
fpp_replaced = fpp.where(~outliers, fpp.median())

# 3. Трансформация: сжимает правый хвост, выбросы перестают быть экстремальными
fpp_log = np.log1p(fpp)                 # логарифм: log(1 + x), безопасен для нулей
fpp_sqrt = np.sqrt(fpp)                 # корень: мягче логарифма
fpp_boxcox = stats.boxcox(fpp + 1)[0]   # box-cox: сам подбирает степень, требует x > 0
fpp_yeo = stats.yeojohnson(fpp)[0]      # yeo-johnson: как box-cox, но умеет 0 и минус

# 4. Кодирование в отдельную категорию: новый признак-флаг «это выброс»
df["FareOutlier"] = outliers.astype(int)

# Все варианты рядом: исходный + 2 метода + 4 трансформации, сеткой в 2 ряда
variants = [
    ("Исходный", fpp), ("Удаление", fpp_dropped),
    ("Замена медианой", fpp_replaced), ("log1p", fpp_log),
    ("Корень", fpp_sqrt), ("Box-Cox", fpp_boxcox), ("Yeo-Johnson", fpp_yeo),
]
fig = make_subplots(rows=2, cols=4, subplot_titles=[name for name, _ in variants])
for i, (name, data) in enumerate(variants):
    fig.add_trace(go.Histogram(x=data, marker_color="#4C78A8"), row=i // 4 + 1, col=i % 4 + 1)
fig.update_layout(template="plotly_white", height=600, showlegend=False, title_text="Методы обработки выбросов FarePerPerson", title_x=0.5)
fig.show()

print("Выбор на дальнейшее: log1p (просто и обратимо),")
print("удаление/замена - такое себе т.к. строк мало, а выбросы не ошибки, а реальные цены.")

Выбросов по правилу |Z| > 3: 13
Выбросов по IQR (за пределами [-17.0, 49.1]): 59
IQR строже: Z-оценка сама страдает от выбросов (они раздувают std).


Выбор на дальнейшее: log1p (просто и обратимо),
удаление/замена - такое себе т.к. строк мало, а выбросы не ошибки, а реальные цены.


**2. Нормализация непрерывных признаков** — Age и FarePerPerson, они разной размерности (в разных масштабах?: годы и фунты

In [5]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

age = df["Age"].dropna()

# Одна и та же Age в трёх видах: сырая, Мин-Макс [0..1], Z-нормализация (среднее 0, std 1)
demo = pd.DataFrame({
    "Age исходный": age.describe(),
    "Age Мин-Макс": pd.Series(MinMaxScaler().fit_transform(age.to_frame())[:, 0]).describe(),
    "Age Z-нормализация": pd.Series(StandardScaler().fit_transform(age.to_frame())[:, 0]).describe(),
}).round(2)
display(demo)

print("Age (0..80) и FarePerPerson (0..~130) - нужна нормализация")
print("без неё линейные модели и KNN будут считать признак с большими числами «важнее».")
print("Деревьям (RF, XGBoost, CatBoost) масштаб безразличен.")
print("Берём Z-нормализацию (StandardScaler): стандартный выбор, устойчивее Мин-Макс к выбросам.")

,Age исходный,Age Мин-Макс,Age Z-нормализация
count,714.00,714.00,714.00
mean,29.70,0.37,0.00
std,14.53,0.18,1.00
min,0.42,0.00,-2.02
25%,20.12,0.25,-0.66
50%,28.00,0.35,-0.12
75%,38.00,0.47,0.57
max,80.00,1.00,3.47


Age (0..80) и FarePerPerson (0..~130) - нужна нормализация
без неё линейные модели и KNN будут считать признак с большими числами «важнее».
Деревьям (RF, XGBoost, CatBoost) масштаб безразличен.
Берём Z-нормализацию (StandardScaler): стандартный выбор, устойчивее Мин-Макс к выбросам.


**3. Отбор признаков.** На старт берём встроенную важность признаков моделей на основе деревьев решений (tree-based). Почему именно Random Forest:

- это ансамбль из 100 деревьев — важность усредняется, результат стабильнее, чем у одного дерева;
- деревья ловят нелинейности и связки признаков (Sex + Age), которые корреляция не видит;
- не требует масштабирования и почти не требует подготовки данных;
- важность достаётся бесплатно — модель всё равно есть в нашем бенчмарке.

Для контроля добавляем в матрицу заведомо случайный признак RandomNoise: важность — величина относительная, и шум даёт «нулевую отметку». Всё, что не важнее шума, — кандидат на выброс.

permutation importance вроде как честнее, но замороченней.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Минимальная подготовка только для демо: пол в 0/1, пропуски возраста — медианой
tmp = df.copy()
tmp["Sex"] = (tmp["Sex"] == "male").astype(int)
tmp["Age"] = tmp["Age"].fillna(tmp["Age"].median())

# Контроль: заведомо случайный признак. Правило простое —
# всё, что по важности не выше шума, кандидат на выброс.
rng = np.random.default_rng(42) # Крутая тема - пробовать разные сиды для рнд-генератора
tmp["RandomNoise"] = rng.normal(size=len(tmp))

feats = ["Pclass", "Sex", "Age", "FarePerPerson", "SibSp", "Parch", "RandomNoise"]
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(tmp[feats], tmp["Survived"])

imp = pd.Series(rf.feature_importances_, index=feats).sort_values()
fig = px.bar(
    imp, orientation="h",
    text_auto=".2f",
    template="plotly_white", height=350,
    title="Важность признаков (Random Forest)",
    labels={"value": "Важность", "index": "Признак"}
)
# Шумовой признак красим красным, остальные — синим
fig.update_traces(marker_color=["#E45756" if f == "RandomNoise" else "#4C78A8" for f in imp.index])
fig.update_layout(title_x=0.5, showlegend=False)
fig.show()

noise_imp = imp["RandomNoise"]
weaker = imp[imp <= noise_imp].drop("RandomNoise").index.tolist()
print(f"Важность шума RandomNoise: {noise_imp:.3f}")
print("Не важнее шума (кандидаты на выброс):", weaker if weaker else "таких нет")

Важность шума RandomNoise: 0.204
Не важнее шума (кандидаты на выброс): ['Parch', 'SibSp', 'Pclass', 'Age']


**4. Кодирование категориальных признаков.** Правило выбора — по числу категорий и типу модели:

- **One-Hot** — категорий мало (Sex — 2, Embarked — 3, Pclass — 3): каждая становится отдельным столбцом 0/1. Наш выбор, используется в коде ниже (`pd.get_dummies`).
- **Label** — категории заменяются номерами 0, 1, 2… Подходит только деревьям (им не важен «порядок» номеров) или когда порядок реален (например, класс билета). Для линейных моделей опасен: модель решит, что S > Q > C.
- **Target** — категория заменяется средней долей выживших в ней. Нужен, когда категорий много (Title, палуба из Cabin) и one-hot раздует таблицу. Опасен утечкой target'а — требует аккуратного расчёта только по train.

**Бенчмарк моделей** (код ниже):

In [8]:
# --- Подготовка данных ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('Titanic-Dataset.csv')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Решения из блоков выше: цена на человека вместо сырого Fare + log1p против перекоса
df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
df['FarePerPersonLog'] = np.log1p(df['Fare'] / df['TicketGroupSize'])

# Переводим категориальные признаки в числа (One-Hot Encoding)
df_ml = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Отбираем признаки для обучения и целевую переменную
X = df_ml[['Pclass', 'Age', 'FarePerPersonLog', 'FamilySize', 'Sex_male', 'Embarked_Q', 'Embarked_S']]
y = df_ml['Survived']

X = X.fillna(0) # Ну да, такое себе, но делать всё совсем по-взрослому вот так сходу чёт тяжело даётся

# Разделяем на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Масштабирование признаков (критично для линейных моделей и KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Модели ---
# Dummy — «пустышка», всегда отвечает самым частым классом («все погибли»).
# Это точка отсчёта (baseline): модель, которая не выучила ничего.
models = {
    "Dummy (все погибли)": DummyClassifier(strategy="most_frequent"),
    "Linear (Logistic Regression)": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(iterations=150, depth=4, random_state=42, verbose=0)
}

# --- Обучение и оценка ---
results = {}
for name, model in models.items():
    # Для логистической регрессии и KNN используем отмасштабированные данные
    if name in ["Linear (Logistic Regression)", "KNN"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    # Метрики: Accuracy — доля верных ответов в целом;
    # Precision — из предсказанных «выжил» сколько выжили на самом деле;
    # Recall — из реально выживших скольких модель нашла;
    # F1 — баланс Precision и Recall (их гармоническое среднее).
    # zero_division=0: Dummy никого не называет выжившим, без этого precision даст ошибку деления
    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds, zero_division=0),
    }

# --- Таблица результатов, отсортирована по убыванию F1 (сводная метрика) ---
sorted_results = sorted(results.items(), key=lambda x: x[1]["F1"], reverse=True)
df_results = pd.DataFrame([{"Модель": name, **metrics} for name, metrics in sorted_results])
for col in ["Accuracy", "Precision", "Recall", "F1"]:
    df_results[col] = df_results[col].map('{:.2%}'.format)
df_results

,Модель,Accuracy,Precision,Recall,F1
0,XGBoost,82.12%,80.33%,71.01%,75.38%
1,Linear (Logistic Regression),80.45%,77.42%,69.57%,73.28%
2,KNN,79.33%,75.81%,68.12%,71.76%
3,CatBoost,79.89%,81.13%,62.32%,70.49%
4,Decision Tree,78.21%,74.19%,66.67%,70.23%
5,Random Forest,78.21%,81.25%,56.52%,66.67%
6,Dummy (все погибли),61.45%,0.00%,0.00%,0.00%


1. Выбросы: ищем по IQR, чиним трансформацией — для FarePerPerson берём log1p; удаление/замена не нужны (выбросы — реальные цены, не ошибки).
2. Нормализация: нужна (Age и FarePerPerson в разных масштабах) — Z-нормализация (StandardScaler), но только для линейных моделей и KNN.
3. Отбор признаков — важность признаков Random Forest с контрольным шумовым признаком RandomNoise; топ ожидаемо у Sex, Age и нашего выведенного признака FarePerPerson, а всё, что не важнее шума, — кандидат на выброс.
4. Кодирование категорий — One-Hot (категорий мало); Label — только для моделей на основе деревьев, Target — прибережём для Title/Cabin.

Бенчмарк считает четыре метрики и включает baseline-пустышку Dummy («все погибли»). Строка Dummy показывает, почему Accuracy здесь недостаточно: не выучив ничего, пустышка получает Accuracy ~62% (доля класса «погиб»), но F1 = 0 — по F1 её бесполезность видна сразу. Поэтому смотрим на Recall (скольких выживших нашли), Precision (сколько из найденных — настоящие) и сортируем по F1 — их балансу. Настоящие модели идут плотной группой, явного фаворита нет — значит, дальше выигрыш надо искать не в переборе моделей, а в признаках: Title, категории FamilySize.